# Circuit Visualization: Quantum Hardware-Efficient Ansatz

This notebook visualizes the quantum circuits used in barren plateau experiments.

**Objectives:**
- Visualize the hardware-efficient ansatz structure
- Inspect circuit depth and gate count
- Compare circuits at different layer depths
- Understand parameterization and entanglement structure

In [ ]:
import cirq
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Add src to path
sys.path.append(str(Path.cwd().parent / 'src'))

from models.quantum_circuit import QuantumCircuit

plt.rcParams['figure.figsize'] = (14, 8)

## 1. Basic Circuit Structure

In [ ]:
# Create a simple circuit
qc = QuantumCircuit(n_qubits=4, n_layers=2)
circuit = qc.get_circuit()

print("Circuit Diagram:")
print(circuit)
print(f"\nNumber of qubits: {qc.n_qubits}")
print(f"Number of layers: {qc.n_layers}")
print(f"Total operations: {len(list(circuit.all_operations()))}")

## 2. Data Encoding Circuit

In [ ]:
# Visualize data encoding
encoding_circuit = qc.create_data_encoding_circuit()

print("Data Encoding Circuit:")
print(encoding_circuit)
print(f"\nData symbols: {qc.get_data_symbols()}")
print(f"Number of data parameters: {len(qc.get_data_symbols())}")

## 3. Variational Circuit

In [ ]:
# Visualize variational circuit
var_circuit = qc.create_variational_circuit()

print("Variational Circuit:")
print(var_circuit)
print(f"\nVariational symbols (first 10): {qc.get_variational_symbols()[:10]}")
print(f"Total variational parameters: {len(qc.get_variational_symbols())}")

## 4. Circuit Depth Analysis

In [ ]:
# Analyze circuits at different depths
depths = [2, 4, 6, 8]
n_qubits = 4

stats = []
for depth in depths:
    qc_temp = QuantumCircuit(n_qubits=n_qubits, n_layers=depth)
    circuit_temp = qc_temp.get_circuit()
    
    n_ops = len(list(circuit_temp.all_operations()))
    n_params = len(qc_temp.get_variational_symbols())
    
    stats.append({
        'depth': depth,
        'operations': n_ops,
        'parameters': n_params
    })

# Plot statistics
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot([s['depth'] for s in stats], 
             [s['operations'] for s in stats], 
             'o-', linewidth=2, markersize=8)
axes[0].set_xlabel('Circuit Depth (Layers)', fontsize=12)
axes[0].set_ylabel('Number of Operations', fontsize=12)
axes[0].set_title('Circuit Operations vs Depth', fontsize=13)
axes[0].grid(alpha=0.3)

axes[1].plot([s['depth'] for s in stats], 
             [s['parameters'] for s in stats], 
             's-', color='coral', linewidth=2, markersize=8)
axes[1].set_xlabel('Circuit Depth (Layers)', fontsize=12)
axes[1].set_ylabel('Number of Parameters', fontsize=12)
axes[1].set_title('Trainable Parameters vs Depth', fontsize=13)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nCircuit Statistics:")
print("-" * 50)
for s in stats:
    print(f"Depth {s['depth']:2d}: {s['operations']:3d} ops, {s['parameters']:3d} params")

## 5. Layerwise Circuit Construction

In [ ]:
# Demonstrate layerwise circuit building
qc_layerwise = QuantumCircuit(n_qubits=4, n_layers=4)

print("Layerwise Circuit Construction:\n")
for layer in range(1, 5):
    circuit_partial = qc_layerwise.get_circuit_up_to_layer(layer)
    n_ops = len(list(circuit_partial.all_operations()))
    print(f"Layer {layer}: {n_ops} operations")
    if layer <= 2:
        print(circuit_partial)
        print()

## 6. Entanglement Structure

In [ ]:
# Analyze CNOT (entangling) gates
qc_ent = QuantumCircuit(n_qubits=4, n_layers=3)
circuit_ent = qc_ent.get_circuit()

# Count CNOTs
cnot_count = 0
cnot_pairs = []

for op in circuit_ent.all_operations():
    if isinstance(op.gate, cirq.CNotPowGate) or 'CNOT' in str(op.gate):
        cnot_count += 1
        qubits = [str(q) for q in op.qubits]
        cnot_pairs.append(qubits)

print(f"Total CNOT gates: {cnot_count}")
print(f"CNOTs per layer: {cnot_count / qc_ent.n_layers:.1f}")
print(f"\nEntanglement pattern (linear topology):")
for i, pair in enumerate(cnot_pairs[:6]):  # Show first 6
    print(f"  CNOT {i+1}: {pair[0]} → {pair[1]}")

## 7. Local vs Global Cost Operators

In [ ]:
# Compare local and global cost operators
qc_cost = QuantumCircuit(n_qubits=4, n_layers=2)

global_ops = qc_cost.create_readout_operators(local_cost=False)
local_ops = qc_cost.create_readout_operators(local_cost=True)

print("Global Cost Function:")
print(f"  Number of operators: {len(global_ops)}")
print(f"  Operator: {global_ops}")

print("\nLocal Cost Function:")
print(f"  Number of operators: {len(local_ops)}")
print(f"  Operators (one per qubit): {local_ops}")

print("\n" + "="*60)
print("Local cost measures each qubit independently")
print("Global cost measures collective qubit state")
print("="*60)

## 8. Parameter Count by Architecture

In [ ]:
# Compare different architectures
qubit_counts = [2, 4, 6, 8]
layer_counts = [2, 4, 6, 8]

param_matrix = np.zeros((len(qubit_counts), len(layer_counts)))

for i, n_q in enumerate(qubit_counts):
    for j, n_l in enumerate(layer_counts):
        qc_temp = QuantumCircuit(n_qubits=n_q, n_layers=n_l)
        param_matrix[i, j] = len(qc_temp.get_variational_symbols())

# Plot heatmap
plt.figure(figsize=(10, 6))
im = plt.imshow(param_matrix, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, label='Number of Parameters')
plt.xticks(range(len(layer_counts)), layer_counts)
plt.yticks(range(len(qubit_counts)), qubit_counts)
plt.xlabel('Number of Layers', fontsize=12)
plt.ylabel('Number of Qubits', fontsize=12)
plt.title('Trainable Parameters by Architecture', fontsize=14)

# Add text annotations
for i in range(len(qubit_counts)):
    for j in range(len(layer_counts)):
        text = plt.text(j, i, int(param_matrix[i, j]),
                       ha="center", va="center", color="black", fontsize=10)

plt.tight_layout()
plt.show()

print("\nParameter scaling: 2 × n_qubits × n_layers")
print("(RY and RZ rotation per qubit per layer)")

## 9. Circuit Simulation Example

In [ ]:
# Simulate a simple circuit
qc_sim = QuantumCircuit(n_qubits=4, n_layers=2)
circuit_sim = qc_sim.get_circuit()

# Resolve parameters with random values
data_symbols = qc_sim.get_data_symbols()
var_symbols = qc_sim.get_variational_symbols()

resolver = cirq.ParamResolver({
    **{s: np.random.rand() * np.pi for s in data_symbols},
    **{s: np.random.rand() * 2 * np.pi for s in var_symbols}
})

resolved_circuit = cirq.resolve_parameters(circuit_sim, resolver)

# Simulate
simulator = cirq.Simulator()
result = simulator.simulate(resolved_circuit)

# Plot state vector
state_vector = result.final_state_vector
probabilities = np.abs(state_vector)**2

plt.figure(figsize=(12, 4))
plt.bar(range(len(probabilities)), probabilities, alpha=0.7, edgecolor='black')
plt.xlabel('Computational Basis State', fontsize=12)
plt.ylabel('Probability', fontsize=12)
plt.title('Final State Distribution (Random Parameters)', fontsize=13)
plt.xticks(range(0, 16, 2), [f'|{i:04b}⟩' for i in range(0, 16, 2)], rotation=45)
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"State vector dimension: {len(state_vector)} (2^{qc_sim.n_qubits})")
print(f"Total probability: {probabilities.sum():.6f}")

## 10. Conclusion

This notebook demonstrated:
- ✅ Hardware-efficient ansatz structure
- ✅ Linear scaling of parameters with depth: 2 × n_qubits × n_layers
- ✅ CNOT entanglement in linear topology
- ✅ Layerwise circuit construction for incremental training
- ✅ Local vs global cost function operators
- ✅ Circuit simulation and state vector analysis

**Key Insights:**
- Circuit depth directly impacts parameter count and barren plateau susceptibility
- Local cost functions measure individual qubits (4 operators)
- Global cost functions measure collective state (1 operator)
- Entanglement structure follows linear nearest-neighbor topology